# MiniMax H3 一键启动

首次打开 Notebook 只需运行下面的启动单元。它会恢复 SSH、安装 H3 加速节点、启动 ComfyUI，并在后台断点下载模型。

SSH 命令由 AMD Profile 提供；ComfyUI 可通过 `ssh -L 8188:127.0.0.1:8188 root@实例IP -p 端口` 访问。

In [ ]:
!bash ../deploy/bootstrap.sh

## 状态检查

可重复运行，不会重复启动服务或重复下载已完成模型。

In [ ]:
!source ../deploy/config.sh; echo '--- SSH ---'; pgrep -x sshd >/dev/null && echo ready || echo unavailable; echo '--- ComfyUI ---'; curl -sS -o /dev/null -w 'HTTP %{http_code}\n' --max-time 5 http://127.0.0.1:$H3_COMFYUI_PORT/ || true; echo '--- 模型 ---'; du -sh $H3_MODELS_DIR 2>/dev/null || true; find $H3_MODELS_DIR -name '*.safetensors.complete' -printf '%P\n' 2>/dev/null; echo '--- 日志 ---'; tail -n 8 $H3_LOG_DIR/bootstrap.log 2>/dev/null || true; tail -n 8 $H3_LOG_DIR/models.log 2>/dev/null || true

## 可选：参考图生视频

参考图权重约 20GB，当前模板的 `/workspace` 容量不足以持久保存完整 H3 模型和该权重。只有确认可用空间足够时再运行。

In [ ]:
!REF2VA=1 bash ../deploy/download_models_modelscope.sh

## 手动重启 ComfyUI

正常情况下由守护进程自动恢复；只有需要手动刷新进程时使用。

In [ ]:
!source ../deploy/config.sh; if [ -f $H3_RUNTIME_DIR/comfyui-supervisor.pid ]; then kill $(cat $H3_RUNTIME_DIR/comfyui-supervisor.pid) 2>/dev/null || true; fi; if [ -f $H3_RUNTIME_DIR/comfyui.pid ]; then kill $(cat $H3_RUNTIME_DIR/comfyui.pid) 2>/dev/null || true; fi; bash ../deploy/start_comfyui.sh